# encoder-decoder-symmetric — worked example 2: ConvTranspose2d Decoder Mirror to a Strided-Conv Encoder

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `encoder-decoder-symmetric`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When the encoder uses strided Conv2d (stride=2) instead of MaxPool for downsampling, the natural symmetric decoder uses ConvTranspose2d(stride=2) instead of Upsample+Conv. Each ConvTranspose2d layer is the 'adjoint' of its encoder counterpart: same kernel size, same stride, channels reversed. This pattern is typical in VAEs and DCGAN generators.

## Worked solution

We build a two-stage encoder using strided Conv2d and a matching decoder using ConvTranspose2d.

**Encoder:** Conv2d(1, 16, k=4, s=2, p=1) then Conv2d(16, 32, k=4, s=2, p=1). Each layer halves H and W. With padding=1 and k=4, s=2, the output is `(H - 4 + 2*1) / 2 + 1 = H/2`. So 32×32 → 16×16 → 8×8.

**Decoder:** ConvTranspose2d(32, 16, k=4, s=2, p=1) then ConvTranspose2d(16, 1, k=4, s=2, p=1). ConvTranspose with k=4, s=2, p=1 maps H → 2H, exactly reversing the encoder. So 8×8 → 16×16 → 32×32.

**Shape formula check:** ConvT output = (H-1)*s - 2*p + k = (H-1)*2 - 2 + 4 = 2H. This exactly inverts the Conv formula.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(13)

class StridedAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: strided conv downsample
        self.enc1 = nn.Conv2d(1,  16, kernel_size=4, stride=2, padding=1)
        self.enc2 = nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1)
        # Decoder: ConvTranspose2d upsample (adjoint of encoder)
        self.dec1 = nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(16,  1, kernel_size=4, stride=2, padding=1)
        self.act  = nn.ReLU()

    def forward(self, x):
        h = self.act(self.enc1(x))
        h = self.act(self.enc2(h))
        h = self.act(self.dec1(h))
        return self.dec2(h)

model = StridedAutoencoder()
model.eval()

x = t.randn(3, 1, 32, 32)
out = model(x)
print(f"Input:  {tuple(x.shape)}")
print(f"Output: {tuple(out.shape)}")
assert out.shape == x.shape, f"Expected {x.shape}, got {out.shape}"

# Verify intermediate shapes
with t.no_grad():
    h1 = model.act(model.enc1(x))
    h2 = model.act(model.enc2(h1))
    d1 = model.act(model.dec1(h2))
    print(f"After enc1: {tuple(h1.shape)}  (expect (3,16,16,16))")
    print(f"After enc2: {tuple(h2.shape)}  (expect (3,32,8,8))")
    print(f"After dec1: {tuple(d1.shape)}  (expect (3,16,16,16))")
assert h1.shape == (3, 16, 16, 16)
assert h2.shape == (3, 32, 8, 8)
assert d1.shape == (3, 16, 16, 16)
print("All intermediate shapes correct.")